# AIoT Project

In [ ]:
import os

# basic data engineering
import pandas as pd
import numpy as np
import scipy

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# db
import pymongo

# configs & other
import yaml
from tqdm.notebook import tqdm_notebook
from datetime import datetime
from time import time

from psynlig import pca_explained_variance_bar

# utils processing
from utils import sliding_window_pd
from utils import apply_filter
from utils import filter_instances
from utils import flatten_instances_df
from utils import df_rebase
from utils import rename_df_column_values

# utils visualization
from utils_visual import plot_instance_time_domain
from utils_visual import plot_instance_3d
from utils_visual import plot_np_instance
from utils_visual import plot_heatmap
from utils_visual import plot_scatter_pca

%load_ext autoreload
%autoreload 2

Start time of execution

In [ ]:
time_start = time()

## Load configuration

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

In [ ]:
client = pymongo.MongoClient(config["client"])

In [ ]:
db = client[config["db"]]
coll = db[config["col"]]

In [ ]:
found_keys = coll.distinct("label")
print("Existing DB keys:", found_keys)

## Load data

In [ ]:
# Fetch all records from the collection
cursor = coll.find({})
data_list = list(cursor)

# Load into DataFrame and drop the MongoDB '_id'
df = pd.DataFrame(data_list)
if not df.empty:
    df = df.drop(columns=['_id'])
    print(f"Successfully loaded {len(df)} total instances across all subjects.")
else:
    print("No data found.")


## Explore the nature of the data

In [ ]:
# Calculate the number of samples for each instance 
# We take the length of the first list inside the 'data' dictionary
df['num_samples'] = df['data'].apply(lambda x: len(next(iter(x.values()))))

# Calculate time length in seconds (samples / sampling rate)
df['time_length_sec'] = df['num_samples'] / df['sr']

# Calculate time length in minutes for easier reading
df['time_length_min'] = df['time_length_sec'] / 60

# Preview the new columns
df[['gesture_id', 'num_samples', 'time_length_sec', 'time_length_min']].head()

In [ ]:
# Group by gesture class and sum the total time length
time_per_class = df.groupby('gesture_id')['time_length_min'].sum().reset_index()
time_per_class = time_per_class.rename(columns={'time_length_min': 'total_time_min'})

# Display the aggregated data
print(time_per_class)

In [ ]:
# Set the figure size
plt.figure(figsize=(10, 6))

# Create the barplot
sns.barplot(data=time_per_class, x='gesture_id', y='total_time_min', palette='viridis')

# Add titles and labels
plt.title('Total Time-Length of Collected Instances per Class', fontsize=14)
plt.xlabel('Gesture Class', fontsize=12)
plt.ylabel('Total Time (Minutes)', fontsize=12)

# Rotate the x-axis labels if they are long
plt.xticks(rotation=45, ha='right')

# Adjust layout to prevent clipping
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Check for missing values in the raw dataset before segmentation
print("Checking for missing values and unequal lengths in the raw continuous data...")
missing_found = False

for index, row in df.iterrows():
    # Convert each array to a pandas Series first. 
    # This forces Pandas to accept unequal lengths by padding the shorter ones with NaNs.
    instance_df = pd.DataFrame({k: pd.Series(v) for k, v in row['data'].items()})
    
    if instance_df.isnull().values.any():
        #print(f"Missing values or unequal lengths found in row {index} (Gesture: {row['gesture_id']}) - Fixing automatically!")
        missing_found = True
        
        # FIXING THE DATA:
        # 1. Interpolate (draw a line) to fill any internal missing packets
        instance_df = instance_df.interpolate(method='linear')
        
        # 2. Drop any remaining NaNs at the very end (truncates all arrays to the length of the shortest one)
        instance_df = instance_df.dropna()
        
        # 3. Save the cleaned data back into our main DataFrame
        df.at[index, 'data'] = instance_df.to_dict('list')

if missing_found:
    print("All missing values and unequal lengths have been successfully cleaned!")
else:
    print("No missing values found in the raw data!")

In [ ]:
def plot_instance_time_domain(df: pd.DataFrame):
    """Visualizes the movement instance to a plot in time domain.

    Args:
        df: The DataFrame to be visualized in time domain.

    Returns:

    """
    df.plot(figsize=(20, 10), linewidth=2, fontsize=20).legend(fontsize=18)
    plt.xlabel('Sample', fontsize=20)
    plt.ylabel('Axes', fontsize=20)


# 1. Extract a continuous instance from the dataframe (e.g., the first recording)
sample_idx = 30
instance_df = pd.DataFrame(df.iloc[sample_idx]['data'])
gesture_id = df.iloc[sample_idx]['gesture_id'] # Get the gesture ID for the title

# 2. Separate accelerometer and gyroscope columns
acc_cols = [col for col in instance_df.columns if 'acc' in col.lower()]
gyr_cols = [col for col in instance_df.columns if 'gyr' in col.lower()]

# Fallback in case columns aren't explicitly named 'acc'/'gyr'
if not acc_cols:
    acc_cols = instance_df.columns[:3] 
if not gyr_cols:
    gyr_cols = instance_df.columns[3:6] 

# 3. Plot Accelerometer channels (Subplot 1)
plot_instance_time_domain(instance_df[acc_cols])
plt.title(f'Accelerometer Channels - Gesture: {gesture_id}', fontsize=22)
plt.show()

# 4. Plot Gyroscope channels (Subplot 2)
plot_instance_time_domain(instance_df[gyr_cols])
plt.title(f'Gyroscope Channels - Gesture: {gesture_id}', fontsize=22)
plt.show()

In [ ]:
from utils import apply_filter

print("Applying lowpass filter to the dataset using utils.py...")

# Extract filter parameters from your config.yml
filter_order = config['filter']['order']
cutoff_freq = config['filter']['wn'] 
filter_type = config['filter']['type']

def filter_session_data(instance_data, sr):
    """
    Wrapper to apply utils.apply_filter to the dictionary of axes,
    handling the Nyquist frequency normalization.
    """
    filtered_data = {}
    
    # 1. Calculate the Nyquist frequency (half of the sampling rate)
    nyquist_freq = 0.5 * sr
    
    # 2. Normalize the cutoff frequency to be between 0 and 1
    normalized_wn = cutoff_freq / nyquist_freq 
    
    for axis_name, values in instance_data.items():
        # 3. Call your existing apply_filter function
        filtered_array = apply_filter(
            arr=values, 
            order=filter_order, 
            wn=normalized_wn,  # MUST use the normalized frequency here
            filter_type=filter_type
        )
        filtered_data[axis_name] = filtered_array.tolist()
        
    return filtered_data

# Apply the filter row by row, respecting the 'sr' of each session
df['filtered_data'] = df.apply(
    lambda row: filter_session_data(row['data'], row['sr']), 
    axis=1
)

print("Filtering complete!")

# Preview the results to verify
df[['gesture_id', 'sr', 'data', 'filtered_data']].head(3)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from utils_visual import plot_instance_time_domain

# 1. Select a single instance (index 0)
sample_idx = 0
raw_data_dict = df.iloc[sample_idx]['data']
filtered_data_dict = df.iloc[sample_idx]['filtered_data']
gesture_label = df.iloc[sample_idx]['gesture_id']

# 2. Convert both data dictionaries into Pandas DataFrames
df_raw = pd.DataFrame(raw_data_dict)
df_filtered = pd.DataFrame(filtered_data_dict)

# 3. Slice the data to view only a specific time frame (e.g., first 300 samples)
zoom_samples = 300 
df_raw_zoomed = df_raw.iloc[:zoom_samples]
df_filtered_zoomed = df_filtered.iloc[:zoom_samples]

# 4. Plot the RAW data
print(f"Plotting RAW data for gesture: {gesture_label} (First {zoom_samples} samples)")
plot_instance_time_domain(df_raw_zoomed)
plt.title(f"Raw Signal Zoomed In - {gesture_label}", fontsize=24)
plt.show()

# 5. Plot the FILTERED data
print(f"Plotting FILTERED data for gesture: {gesture_label} (First {zoom_samples} samples)")
plot_instance_time_domain(df_filtered_zoomed)
plt.title(f"Filtered Signal Zoomed In - {gesture_label}", fontsize=24)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
GESTURE_COL = 'gesture_id' 
DATA_COL = 'filtered_data' 
SUBJECT_COL = 'user'  
# ---------------------

# Get lists of unique gestures and unique subjects
unique_gestures = df[GESTURE_COL].unique()
unique_subjects = df[SUBJECT_COL].unique()

# --- NEW: Strict RGB Palette for 3 Subjects ---
rgb_colors = ['red', 'green', 'blue']

# Assign a specific RGB color to each unique subject safely
subject_colors = {}
for i, subject in enumerate(unique_subjects):
    # Use modulo (%) to wrap around safely
    subject_colors[subject] = rgb_colors[i % len(rgb_colors)]

for gesture in unique_gestures:
    print(f"\n{'='*60}")
    print(f"GESTURE: {str(gesture).upper()}")
    print(f"{'='*60}")
    
    # 1. Get all rows for this specific gesture
    gesture_rows = df[df[GESTURE_COL] == gesture]
    
    # 2. Create a figure with two 3D subplots side-by-side
    fig = plt.figure(figsize=(16, 8))
    
    # --- Setup Accelerometer Subplot (Left) ---
    ax_acc = fig.add_subplot(121, projection='3d')
    ax_acc.set_title(f"Accelerometer 3D Trajectory\n{gesture} (By Subject)", pad=20, fontsize=14)
    ax_acc.set_xlabel("acc_x")
    ax_acc.set_ylabel("acc_y")
    ax_acc.set_zlabel("acc_z")
    
    # --- Setup Gyroscope Subplot (Right) ---
    ax_gyr = fig.add_subplot(122, projection='3d')
    ax_gyr.set_title(f"Gyroscope 3D Trajectory\n{gesture} (By Subject)", pad=20, fontsize=14)
    ax_gyr.set_xlabel("gyr_x")
    ax_gyr.set_ylabel("gyr_y")
    ax_gyr.set_zlabel("gyr_z")
    
    # 3. Loop through EVERY session
    for _, row in gesture_rows.iterrows():
        session_data_dict = row[DATA_COL]
        subject_id = row[SUBJECT_COL]  
        
        # Look up the designated color for this subject
        color = subject_colors[subject_id]
        
        # Convert dictionary to DataFrame
        instance_df = pd.DataFrame({k: pd.Series(v) for k, v in session_data_dict.items()})
        
        # Plot trajectories using the SUBJECT'S custom RGB color
        ax_acc.plot(instance_df["acc_x"], instance_df["acc_y"], instance_df["acc_z"], color=color, alpha=0.3)
        ax_gyr.plot(instance_df["gyr_x"], instance_df["gyr_y"], instance_df["gyr_z"], color=color, alpha=0.3)

    # 4. Create a custom legend
    legend_elements = [Line2D([0], [0], color=color, lw=4, label=f'Subject {sub}') 
                       for sub, color in subject_colors.items()]
    
    # Add the legend to the right side of the figure
    ax_gyr.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1))
        
    # Adjust layout and display
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
DATA_COL = 'filtered_data' 
# ---------------------

print("Concatenating all sessions across all gestures. This might take a moment...")

# List to hold the DataFrame of every single session in the dataset
all_sessions_dfs = []

# 1. Loop through EVERY row in the entire dataframe
for _, row in df.iterrows():
    session_data_dict = row[DATA_COL]
    
    # Convert the dictionary to a DataFrame, safely handling unequal array lengths
    instance_df = pd.DataFrame({k: pd.Series(v) for k, v in session_data_dict.items()})
    
    all_sessions_dfs.append(instance_df)
    
# 2. Concatenate all sessions into one massive continuous DataFrame
global_df = pd.concat(all_sessions_dfs, ignore_index=True)

# 3. Calculate the Pearson correlation matrix for the entire dataset
global_corr_matrix = global_df.corr()

# 4. Create a mask for the upper triangle
# np.ones_like creates a matrix of True values the same shape as our correlation matrix
# np.triu isolates just the upper triangle of that matrix
mask = np.triu(np.ones_like(global_corr_matrix, dtype=bool))

# 5. Plot the Global Heatmap
plt.figure(figsize=(10, 8))

# Add the 'mask=mask' argument to hide the upper triangle
sns.heatmap(global_corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1,
            linewidths=0.5, square=True, cbar_kws={"shrink": .8})

plt.title("Global Sensor Correlation Heatmap\n(All Gestures & All Sessions Combined)", pad=20, fontsize=16)

# Adjust layout and display
plt.tight_layout()
plt.show()

print(f"Total data points analyzed: {len(global_df)}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
DATA_COL = 'filtered_data' 
# ---------------------

print("Concatenating all sessions across all gestures. This might take a moment...")

# List to hold the DataFrame of every single session in the dataset
all_sessions_dfs = []

# 1. Loop through EVERY row in the entire dataframe
for _, row in df.iterrows():
    session_data_dict = row[DATA_COL]
    
    # Convert the dictionary to a DataFrame, safely handling unequal array lengths
    instance_df = pd.DataFrame({k: pd.Series(v) for k, v in session_data_dict.items()})
    
    all_sessions_dfs.append(instance_df)
    
# 2. Concatenate all sessions into one massive continuous DataFrame
global_df = pd.concat(all_sessions_dfs, ignore_index=True)

# 3. Create a figure with two subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Setup Accelerometer Boxplot (Left) ---
# We isolate only the accelerometer columns so they scale correctly
sns.boxplot(data=global_df[['acc_x', 'acc_y', 'acc_z']], ax=axes[0], palette="Blues")
axes[0].set_title("Accelerometer Features Variance\n(All Sessions Combined)", pad=15, fontsize=14)
axes[0].set_ylabel("Amplitude (Linear Acceleration)")
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# --- Setup Gyroscope Boxplot (Right) ---
# We isolate only the gyroscope columns so they scale correctly
sns.boxplot(data=global_df[['gyr_x', 'gyr_y', 'gyr_z']], ax=axes[1], palette="Reds")
axes[1].set_title("Gyroscope Features Variance\n(All Sessions Combined)", pad=15, fontsize=14)
axes[1].set_ylabel("Amplitude (Angular Velocity)")
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

# Adjust layout and display
plt.tight_layout()
plt.show()

print(f"Total data points visualized: {len(global_df)}")

In [ ]:
import pandas as pd
from utils import sliding_window_pd

print("Starting sliding window segmentation...")

# 1. Extract parameters from your config.yml
ws = config['sliding_window']['ws']
overlap = config['sliding_window']['overlap']
w_type = config['sliding_window']['w_type']
w_center = config['sliding_window']['w_center']

# Initialize our unified lists
all_windows = [] # This will hold our X (list of Window DataFrames)
all_labels = []  # This will hold our y (list of target gesture_ids)

# 2. Iterate over each recording session independently
for index, row in df.iterrows():
    gesture_label = row['gesture_id']
    filtered_data_dict = row['filtered_data']
    
    # Convert the filtered dictionary back into a Pandas DataFrame
    session_df = pd.DataFrame(filtered_data_dict)
    
    # 3. Apply your existing sliding window function
    # This returns a list of small DataFrames (the windows) for this specific session
    session_windows = sliding_window_pd(
        df=session_df,
        ws=ws,
        overlap=overlap,
        w_type=w_type,
        w_center=w_center,
        print_stats=False # Keep it False so it doesn't flood your Jupyter output
    )
    
    # 4. Add these new windows to our unified master list
    all_windows.extend(session_windows)
    
    # 5. Add the exact same number of labels to our labels list
    all_labels.extend([gesture_label] * len(session_windows))

print("Segmentation complete!")
print(f"Total windows extracted (X): {len(all_windows)}")
print(f"Total labels extracted (y): {len(all_labels)}")

In [ ]:
"""
1. label encoding
2. split train/test
3. transform (standardization - scaling)
kai xwrizoume ta arxeia kai blepoyume ti kratame sto kathena
na balw class distribution se shared arxeio
check if windows of the smae gesture look very different
"""

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Convert the list of labels to a Pandas Series for easy analysis
labels_series = pd.Series(all_labels)

# 2. Calculate raw counts and percentages
class_counts = labels_series.value_counts()
class_percentages = labels_series.value_counts(normalize=True) * 100

# 3. Print a clean summary table
imbalance_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_percentages
})

print("Class Distribution After Window Segmentation:\n")
print(imbalance_df.round(2))
print("\n" + "="*60 + "\n")

# 4. Plot the distribution
plt.figure(figsize=(12, 6))

# Create the bar plot (sorted by default due to value_counts)
ax = sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')

plt.title("Class Distribution After Window Segmentation", fontsize=16, pad=20)
plt.xlabel("Gesture Class", fontsize=14, labelpad=10)
plt.ylabel("Number of Windows", fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)

# 5. Add the Count and Percentage text on top of each bar
for i, p in enumerate(ax.patches):
    height = p.get_height()
    percentage = class_percentages.iloc[i]
    
    # Format the text (e.g., "450\n(20.5%)")
    annotation_text = f'{int(height)}\n({percentage:.1f}%)'
    
    ax.annotate(annotation_text, 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='bottom', 
                fontsize=11, fontweight='bold', color='black',
                xytext=(0, 5), textcoords='offset points')

# Expand y-axis slightly to make room for the text annotations at the top
plt.ylim(0, class_counts.max() * 1.15)

# Adjust layout to prevent the rotated X-labels from getting cut off
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------
# Window Flattening & Label Encoding
# ---------------------------------------------
from utils import encode_labels

print("Starting window flattening...")

# 1. Flatten the list of window DataFrames into a single 2D DataFrame (Samples x Features)
# Each window (e.g., 100 rows x 6 sensor axes) becomes one long row (e.g., 600 features)
flattened_df = flatten_instances_df(all_windows)

print(f"Shape of the flattened dataset (X): {flattened_df.shape}")

# 2. Assign the tracked string labels to a new target column 'y'
flattened_df['y'] = all_labels

# 3. Separate features (X) and target (y)
X = flattened_df.drop(columns=['y']).values
y_strings = flattened_df['y'].tolist()

# 4. Encode the string labels (e.g., 'scroll-up-thumb') into integers (e.g., 0, 1, 2...)
y_encoded = encode_labels(y_strings)

print("Flattening and encoding complete!")
print(f"X shape: {X.shape}")
print(f"y shape: {y_encoded.shape}")

# Optional: View the mapping of unique encoded classes
unique_classes = np.unique(y_encoded)
print(f"Unique encoded classes: {unique_classes}")

## Data Processing

## Train/Test split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=True, random_state=42)

## Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

## Classifier - Statistical Learning

### Apply simple classifier

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

### Evaluate simple classifier

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from sklearn.metrics import classification_report

### Apply optimization with Grid Search and/or Cross-validation

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

### Evaluate optimized classifier